# AlphaTensor's 47-term matrix multiplication

**Goal.** Multiply two 4×4 matrices of bits using the published 47-term decomposition. We use ordinary Python addition and multiplication to evaluate the factors, then reduce the sums modulo 2. This computes the matrix product **over** $\mathbb F_2=\{0,1\}$; it does not compute an ordinary integer matrix product.

**Where the 47 terms come from.** AlphaTensor searches for a short decomposition of the matrix multiplication tensor. The 47 coefficient triples below are the *published output of that search*, not something this notebook discovers or learns. Each triple has three length-16 lists of 0/1 coefficients, named `u`, `v`, and `w`. The lists are in row-major order: index 0 means matrix entry (1,1), index 15 means (4,4). The output factors have already been placed in row-major order to match our flattened $C$.

Let $a,b,c\in\{0,1\}^{16}$ be the flattened input and output matrices. For factor $r$ and output position $o$,

$$
\begin{aligned}
L_r &= \sum_{i=0}^{15}u_{ri}a_i, & R_r &= \sum_{j=0}^{15}v_{rj}b_j, &&\text{(1)}\\
p_r &= L_rR_r, &&&&\text{(2)}\\
c_o &= \left(\sum_{r=1}^{47}w_{ro}p_r\right)\bmod 2. &&&&\text{(3)}
\end{aligned}
$$

There are **47 scalar products** $p_r$. The many additions in the linear combinations also cost work.

### Why these 47 factors use modulo 2

AlphaTensor is a search method and is **not limited to modulo 2**. The specific 47-factor data in this notebook was found for $\mathbb F_2$. Its 0/1 coefficients reproduce each coefficient of the matrix product **after reduction modulo 2**. For example, the 47 terms give coefficient 2 for $a_{0,0}b_{2,0}$ in output $c_{0,1}$, where the correct coefficient is 0. This agrees modulo 2 because $2\bmod2=0$; with ordinary integer arithmetic, the unwanted term would remain. Therefore these exact 47 factors multiply binary matrices modulo 2, but cannot be used unchanged for ordinary integer matrix multiplication.

Source: Fawzi et al., *Nature* 610 (2022), DOI [10.1038/s41586-022-05172-4](https://doi.org/10.1038/s41586-022-05172-4), especially Algorithm 1 and Extended Data Fig. 1. Factor data: [authors' repository](https://github.com/google-deepmind/alphatensor), commit `1949163da3bef7e3eb268a3ac015fd1c2dbfc767`, `algorithms/factorizations_f2.npz`. The source license is in `dist/alphatensor-LICENSE.txt` beside this notebook.

In [ ]:
# Eq. (1)–(3): published 0/1 coefficient vectors; each list has 16 entries.
TERMS_47 = [
    {"u": [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "w": [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0], "w": [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], "w": [0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0], "w": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "v": [1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], "v": [0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0], "v": [1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]},
    {"u": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]},
    {"u": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], "v": [1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]},
    {"u": [1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0], "v": [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], "v": [0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1], "v": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], "v": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0]},
    {"u": [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1], "w": [0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0], "w": [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0], "w": [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0], "w": [0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0]},
    {"u": [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], "w": [0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], "v": [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1], "w": [0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1], "w": [0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1], "w": [0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]},
    {"u": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0], "w": [0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], "w": [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], "w": [0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], "v": [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0], "v": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]},
    {"u": [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], "w": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0], "v": [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]},
    {"u": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0], "v": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1], "w": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], "v": [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]},
    {"u": [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "v": [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1], "w": [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0], "v": [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], "v": [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0], "w": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1]},
    {"u": [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1], "v": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1], "w": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]},
    {"u": [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1], "v": [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "w": [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0]},
]
assert len(TERMS_47) == 47
assert all(len(term[key]) == 16 for term in TERMS_47 for key in ('u', 'v', 'w'))
print("Loaded", len(TERMS_47), "published factors as plain 0/1 arrays")

Loaded 47 published factors as plain 0/1 arrays


## 1. Compute the product step by step

First, form the two weighted sums $L_r$ and $R_r$ in Eq. (1). Multiply them to get one $p_r$ in Eq. (2). Add $w_{ro}p_r$ into every output position in Eq. (3), then take `% 2` after the 47 terms. We also implement the textbook 4×4 product for comparison:

$$c_{4i+j}=\left(\sum_{k=0}^{3}a_{4i+k}b_{4k+j}\right)\bmod 2.\tag{4}$$

The loops below do the main work themselves; no matrix multiplication library is used.

In [ ]:
def multiply_factors(a, b, terms, show_steps=False):
    # Eq. (1)–(3): evaluate a list of 0/1 factor triples using + and *.
    assert len(a) == len(b) == 16
    output_sums = [0] * 16
    for step, term in enumerate(terms, start=1):
        # Eq. (1): compute the two linear combinations.
        left = sum(term["u"][i] * a[i] for i in range(16))
        right = sum(term["v"][j] * b[j] for j in range(16))
        # Eq. (2): one scalar multiplication for this factor.
        product = left * right
        # Eq. (3): add this product to its selected output positions.
        for o in range(16):
            output_sums[o] += term["w"][o] * product
        if show_steps and step <= 2:
            selected = [o for o, weight in enumerate(term["w"]) if weight]
            print(f"term {step:2d}: L={left}, R={right}, p={product}, output indices={selected}")
    # Eq. (3): reduce every finished output sum modulo 2.
    return [value % 2 for value in output_sums]


def multiply_direct(a, b):
    # Eq. (4): the textbook 4×4 product, calculated with +, *, and % 2.
    c = [0] * 16
    for i in range(4):
        for j in range(4):
            total = 0
            for k in range(4):
                total += a[4 * i + k] * b[4 * k + j]
            c[4 * i + j] = total % 2
    return c


A = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
B = [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1]
C47 = multiply_factors(A, B, TERMS_47, show_steps=True)
assert C47 == multiply_direct(A, B)
print("4×4 result:", [C47[i:i + 4] for i in range(0, 16, 4)])

term  1: L=1, R=1, p=1, output indices=[0, 1, 4]
term  2: L=3, R=1, p=3, output indices=[3, 4]
4×4 result: [[1, 1, 1, 1], [1, 1, 0, 1], [0, 0, 1, 0], [0, 0, 1, 0]]


## 2. Check the complete factorization identity

One example is not enough to verify an algorithm for every possible input. Consider the coefficient of $a_i b_j$ in output $c_o$. For ordinary 4×4 multiplication, its target value $T_{oij}$ is 1 if that pair belongs in output $o$, and 0 otherwise. The 47 factors reconstruct that coefficient as

$$\widehat T_{oij}=\left(\sum_{r=1}^{47}w_{ro}u_{ri}v_{rj}\right)\bmod 2.\tag{5}$$

We check all $16\times16\times16=4096$ coefficients.

In [ ]:
def coefficient_errors(terms):
    # Eq. (5): count disagreements with all 4096 target coefficients.
    errors = 0
    for o in range(16):
        row, col = divmod(o, 4)
        for i in range(16):
            for j in range(16):
                # Eq. (4): identify the coefficient in the direct product.
                target = int(i // 4 == row and j % 4 == col and i % 4 == j // 4)
                # Eq. (5): sum the contributions of every factor and reduce mod 2.
                found = sum(term["w"][o] * term["u"][i] * term["v"][j]
                            for term in terms) % 2
                errors += found != target
    return errors


errors = coefficient_errors(TERMS_47)
print(f"Coefficient mismatches: {errors} / 4096")
assert errors == 0

Coefficient mismatches: 0 / 4096


## 3. Build two comparison algorithms

The direct method has one product for each choice of output row, output column, and shared index $k$: $4\times4\times4=64$ products. A 2×2 Strassen decomposition has seven products. Applying its factor pattern at both the outer block level and the inner entry level gives $7\times7=49$ products for a 4×4 matrix.

For a 2×2 example, let

$$
A=\begin{bmatrix}a&b\\c&d\end{bmatrix},\qquad
B=\begin{bmatrix}e&f\\g&h\end{bmatrix}.
$$

Direct multiplication gives $C_{11}=ae+bg$, $C_{12}=af+bh$, $C_{21}=ce+dg$, and $C_{22}=cf+dh$, with each sum reduced modulo 2. That requires eight scalar products. Strassen instead forms seven products:

$$
\begin{aligned}
M_1 &= (a+d)(e+h), & M_2 &= (c+d)e,\\
M_3 &= a(f+h),     & M_4 &= d(g+e),\\
M_5 &= (a+b)h,     & M_6 &= (c+a)(e+f),\\
M_7 &= (b+d)(g+h).
\end{aligned}
$$

Then recombine them into the four outputs:

$$
\begin{aligned}
C_{11} &= (M_1+M_4+M_5+M_7)\bmod 2,\\
C_{12} &= (M_3+M_5)\bmod 2,\\
C_{21} &= (M_2+M_4)\bmod 2,\\
C_{22} &= (M_1+M_2+M_3+M_6)\bmod 2.
\end{aligned}
$$

For example, expanding the first output gives

$$M_1+M_4+M_5+M_7
=ae+bg+2(ah+de+dg+dh+bh)
\equiv ae+bg\pmod 2.$$

The repeated terms disappear when the sum is reduced modulo 2. The other three outputs similarly reduce to their direct formulas. Applying the seven-product rule at the block level and again inside each block product gives $7\times7=49$ scalar products for 4×4 matrices.

The `S2` entries below are also plain 0/1 coefficient lists. Because this notebook works modulo 2, a subtraction in Strassen's usual real-number formulas has the same effect as an addition: $-1\equiv1\pmod2$. This 49-factor construction is therefore a **modulo-2** Strassen comparison. We expand outer and inner 2×2 positions into 4×4 row-major positions using ordinary coefficient multiplication.

In [ ]:
def direct_terms():
    # Eq. (4): make one factor for each (output row, output column, k).
    terms = []
    for row in range(4):
        for col in range(4):
            for k in range(4):
                u, v, w = [0] * 16, [0] * 16, [0] * 16
                u[4 * row + k] = 1
                v[4 * k + col] = 1
                w[4 * row + col] = 1
                terms.append({"u": u, "v": v, "w": w})
    return terms


# Eq. (6): seven 2×2 Strassen factors as plain coefficient arrays.
# Positions 0,1,2,3 mean upper-left, upper-right, lower-left, lower-right.
S2 = [
    {"u": [1, 0, 0, 1], "v": [1, 0, 0, 1], "w": [1, 0, 0, 1]},
    {"u": [0, 0, 1, 1], "v": [1, 0, 0, 0], "w": [0, 0, 1, 1]},
    {"u": [1, 0, 0, 0], "v": [0, 1, 0, 1], "w": [0, 1, 0, 1]},
    {"u": [0, 0, 0, 1], "v": [1, 0, 1, 0], "w": [1, 0, 1, 0]},
    {"u": [1, 1, 0, 0], "v": [0, 0, 0, 1], "w": [1, 1, 0, 0]},
    {"u": [1, 0, 1, 0], "v": [1, 1, 0, 0], "w": [0, 0, 0, 1]},
    {"u": [0, 1, 0, 1], "v": [0, 0, 1, 1], "w": [1, 0, 0, 0]},
]


def expand_coefficients(outer, inner):
    # Eq. (7): expand one outer and one inner 2×2 factor into 4×4 positions.
    result = [0] * 16
    for p in range(4):
        for q in range(4):
            row = 2 * (p // 2) + q // 2
            col = 2 * (p % 2) + q % 2
            result[4 * row + col] = outer[p] * inner[q]
    return result


def strassen_49_terms():
    # Eq. (7): seven outer factors times seven inner factors = 49 products.
    terms = []
    for outer in S2:
        for inner in S2:
            terms.append({key: expand_coefficients(outer[key], inner[key])
                          for key in ("u", "v", "w")})
    return terms


TERMS_64 = direct_terms()
TERMS_49 = strassen_49_terms()
assert len(TERMS_64) == 64 and len(TERMS_49) == 49
assert coefficient_errors(TERMS_64) == coefficient_errors(TERMS_49) == 0
assert multiply_factors(A, B, TERMS_49) == C47
print("All three decompositions pass the 4096-coefficient check.")

All three decompositions pass the 4096-coefficient check.


## 4. Experiment: when do fewer products help?

Count additions for a **simple expanded implementation** that skips zero coefficients. A sum of $k$ selected inputs uses $k-1$ additions; an output receiving $q$ products uses $q-1$ additions. We do not share repeated subexpressions, so these are not optimal counts for Strassen or AlphaTensor.

Let one addition cost 1 and one scalar multiplication cost $m$. Our toy operation cost is

$$\operatorname{cost}(m)=mR+A_{\rm input}+A_{\rm output},\tag{8}$$

where $R$ is the factor count. Vary $m$ and compare the methods. This model is **not measured runtime** and excludes the small bookkeeping cost of `% 2`.

In [ ]:
def operation_counts(terms):
    # Eq. (8): additions needed to form nonzero input linear combinations.
    input_additions = sum((sum(term["u"]) - 1) + (sum(term["v"]) - 1)
                          for term in terms)
    # Eq. (8): additions needed to combine terms at each output.
    output_additions = sum(sum(term["w"][o] for term in terms) - 1
                           for o in range(16))
    return len(terms), input_additions, output_additions


methods = {"Direct": TERMS_64, "Strassen": TERMS_49, "AlphaTensor": TERMS_47}
stats = {name: operation_counts(terms) for name, terms in methods.items()}
print("Method       products  input adds  output adds  total adds")
for name, (products, a_in, a_out) in stats.items():
    print(f"{name:12s} {products:8d} {a_in:11d} {a_out:12d} {a_in + a_out:11d}")
print("\nToy cost = m × products + total additions")
print("m       Direct  Strassen  AlphaTensor  lowest")
for m in (1, 5, 10, 20, 50):
    # Eq. (8): change the relative cost of one scalar multiplication.
    costs = {name: m * products + a_in + a_out
             for name, (products, a_in, a_out) in stats.items()}
    print(f"{m:<7d} {costs['Direct']:6d} {costs['Strassen']:9d} "
          f"{costs['AlphaTensor']:12d}  {min(costs, key=costs.get)}")

Method       products  input adds  output adds  total adds
Direct             64           0           48          48
Strassen           49         190          128         318
AlphaTensor        47         202          138         340

Toy cost = m × products + total additions
m       Direct  Strassen  AlphaTensor  lowest
1          112       367          387  Direct
5          368       563          575  Direct
10         688       808          810  Direct
20        1328      1298         1280  AlphaTensor
50        3248      2768         2690  AlphaTensor


## 5. Final independent check against NumPy

Only this last cell uses NumPy, as a reference implementation. For 100 random pairs of binary 4×4 matrices, compare every hand-written algorithm with `numpy.matmul(A, B) % 2`. The 4096-coefficient check above verifies the identity for all inputs; these examples also check that the code executes and indexes the arrays correctly.

**Limitation.** The published 47 factors apply to binary matrix multiplication. Fewer products do not automatically mean faster runtime because the factors need many additions.

In [ ]:
import numpy as np

# Eq. (4): NumPy is used only as the independent reference C = (A @ B) mod 2.
rng = np.random.default_rng(47)
for _ in range(100):
    a_np = rng.integers(0, 2, size=(4, 4), dtype=np.int64)
    b_np = rng.integers(0, 2, size=(4, 4), dtype=np.int64)
    a, b = a_np.ravel().tolist(), b_np.ravel().tolist()
    library_result = (a_np @ b_np) % 2
    for name, terms in methods.items():
        mine = np.array(multiply_factors(a, b, terms)).reshape(4, 4)
        assert np.allclose(mine, library_result), name
    assert np.allclose(np.array(multiply_direct(a, b)).reshape(4, 4), library_result)
print("PASS: 100 random pairs, all methods agree with NumPy.")

PASS: 100 random pairs, all methods agree with NumPy.
